In [32]:
import numpy as np
import pandas as pd
from numpy.linalg import matrix_rank
from numpy.linalg import inv
from numpy.linalg import det
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [33]:
def matrixSolver(X, y, x_ncol, y_ncol, l, bias):
    X = X.reshape(-1, x_ncol)
    # add bias column
    if bias == True:
        X = np.insert(X,0,1,axis=1)
    y = y.reshape(-1, y_ncol)
    n = X.shape[0]
    d = X.shape[1]
    # use ridge regression
    if l > 0:
        if d <= n:
            I = np.eye(d)
            w = inv(X.T@X+l*I)@X.T@y
            print(f"Ridge regression (PRIMAL form)\nw is\n{w}\nMSE is {mean_squared_error(y, X@w)}")
        else:
            I = np.eye(n)
            w = X.T@inv(X@X.T+l*I)@y
            print(f"Ridge regression (DUAL form)\nw is\n{w}\nMSE is {mean_squared_error(y, X@w)}")
        return w
    # uses ordinary least square
    if n > d:
        if not np.isclose(np.linalg.det(X.T@X), 0):
            w = inv(X.T@X)@X.T@y
            print(f"Over-determined({X.shape[0]}x{X.shape[1]}) system")
            print(f"det(X.T@X) = {np.linalg.det(X.T@X)} and w is\n{w}")
            print(f"MSE is {mean_squared_error(y, X@w)}")
        else:
            print("Over-determined system where det(X.T@X) = 0")
            return
    elif n < d:
        if not np.isclose(np.linalg.det(X@X.T), 0):
            w = X.T@inv(X@X.T)@y
            print(f"Under-determined({X.shape[0]}x{X.shape[1]}) system")
            print(f"det(X@X.T) = {np.linalg.det(X@X.T)} and w is\n{w}")
            print(f"MSE is {mean_squared_error(y, X@w)}")
        else:
            print("Under-determined system where det(X@X.T) = 0")
            return
    else:
        if not np.isclose(np.linalg.det(X), 0):
            w = inv(X)@y
            print(f"Even-determined({X.shape[0]}x{X.shape[1]}) system\ndet(X) = {np.linalg.det(X)} and w is\n{w}\nMSE is {mean_squared_error(y, X@w)}")
        else:
            print("Even-determined system where det(X) = 0")
            return
    return w

def polynomialRegression(X, y, x_ncol, y_ncol, deg, lamda, bias):
    X = X.reshape(-1, x_ncol)
    y = y.reshape(-1, y_ncol)
    
    poly_features = PolynomialFeatures(degree=deg)
    P = poly_features.fit_transform(X)
    
    w = matrixSolver(P, y, P.shape[1], y_ncol, lamda, bias)
    return w, P

In [36]:
X = np.array([3,4,5,6,7])
y = np.array([5,4,3,2,1])
X = X.reshape(-1,1)
y = y.reshape(-1,1)
w = matrixSolver(X,y,1,1,0,True)
X_test = np.array([[1,8]])
y_pred = X_test@w
print(y_pred)

Over-determined(5x2) system
det(X.T@X) = 49.99999999999999 and w is
[[ 8.]
 [-1.]]
MSE is 2.871453695004483e-29
[[6.21724894e-15]]


In [40]:
X = np.array([1,2,0,6,1,0,0,5,1,7])
y = np.array([1,2,3,4,5])
X = X.reshape(-1,2)
y = y.reshape(-1,1)
w = matrixSolver(X,y,2,1,0,True)
X_test = np.array([[1,1,3]])
y_pred = X_test@w
print(y_pred)

Over-determined(5x3) system
det(X.T@X) = 158.99999999999986 and w is
[[1.13207547]
 [0.8490566 ]
 [0.33962264]]
MSE is 1.388679245283019
[[3.]]


In [41]:
X = np.array([1,2,2.9])
y = np.array([2,4.1,6.1])
w = matrixSolver(X,y,1,1,0,True)

Over-determined(3x2) system
det(X.T@X) = 5.419999999999997 and w is
[[-0.17509225]
 [ 2.15682657]]
MSE is 0.0007441574415744267


In [39]:
X = np.array([1,2,2.9])
y = np.array([2,4.1,6.1])
w = matrixSolver(X,y,1,1,0.1,True)

Ridge regression (PRIMAL form)
w is
[[0.03832556]
 [2.04765945]]
MSE is 0.007922748024973668
